# DA-Fusion Kaggle Pipeline

Kaggle-submission version of the DA-Fusion experiment. Structure mirrors Victor's `kaggle_cems_pipeline.ipynb` baseline, but:
- **CEMS removed** uses plain ERM training so the comparison with the baseline isolates the effect of DA-Fusion synthetic data only.
- **DA-Fusion section added** diffusion-generated synthetic images are loaded, leakage-filtered (only synthetics whose source image is in the training fold are kept), DINOv2 features extracted, and concatenated to the real training features.

**Required Kaggle datasets (mount in the notebook sidebar before running):**
1. Competition data `csiro-biomass` (auto-mounted when notebook is created from the competition page).
2. DINOv2 weights, e.g. Victor's `darealvictorslorer/dinov2-small-weights` — see `download_dino_weights.py` if making your own.
3. **Your** DA-Fusion synthetic images + `train_augmented.csv` — upload `data/image/augmented/` and `data/tabular/train/train_augmented.csv` as a Kaggle Dataset and point `cfg.synthetic_dir` / `cfg.synthetic_csv` at it.

## 1. Setup & Imports

In [ ]:
import os, math, random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 2. Configuration

Paths below assume the standard Kaggle layout. **Update `cfg.synthetic_dir` / `cfg.synthetic_csv`** to match the Kaggle Dataset slug where you uploaded your DA-Fusion synthetic images.

In [ ]:
cfg = SimpleNamespace(
    # --- Paths ---
    dataset_dir="/kaggle/input/competitions/csiro-biomass",
    dino_weights_dir="/kaggle/input/datasets/darealvictorslorer/dinov2-small-weights/dinov2-small",
    # DA-Fusion synthetic data — UPDATE these after you upload your synthetic images as a Kaggle Dataset
    synthetic_dir="/kaggle/input/da-fusion-synthetic",            # folder containing the synthetic image files
    synthetic_csv="/kaggle/input/da-fusion-synthetic/train_augmented.csv",
    output_dir="/kaggle/working",
    # --- Model ---
    input_dim=384,
    latent_dim=32,
    output_dim=5,
    dropout=0.1,
    # --- Training ---
    epochs=80,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-3,
    seed=42,
    # --- Split ---
    n_splits=5,
    val_fold=0,
    # --- Real-image augmentation (matches Victor's baseline) ---
    use_flip4x=True,          # 4 orientations (identity, hflip, vflip, 180°) on REAL images only
)

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {"Dry_Green_g": 0.1, "Dry_Dead_g": 0.1, "Dry_Clover_g": 0.1,
           "GDM_g": 0.2, "Dry_Total_g": 0.5}

TRAIN_CSV    = os.path.join(cfg.dataset_dir, "train.csv")
TEST_CSV     = os.path.join(cfg.dataset_dir, "test.csv")
TRAIN_IMG_DIR = os.path.join(cfg.dataset_dir, "train")
TEST_IMG_DIR  = os.path.join(cfg.dataset_dir, "test")

print("train.csv:     ", os.path.exists(TRAIN_CSV))
print("test.csv:      ", os.path.exists(TEST_CSV))
print("train img dir: ", os.path.exists(TRAIN_IMG_DIR))
print("test  img dir: ", os.path.exists(TEST_IMG_DIR))
print("dino dir:      ", os.path.exists(cfg.dino_weights_dir))
print("synth dir:     ", os.path.exists(cfg.synthetic_dir))
print("synth csv:     ", os.path.exists(cfg.synthetic_csv))

## 3. Load DINOv2

In [ ]:
print(f"Loading DINOv2 ViT-S/14 from {cfg.dino_weights_dir} ...")
dino = AutoModel.from_pretrained(cfg.dino_weights_dir).eval().to(device)
for p in dino.parameters():
    p.requires_grad_(False)

# Smoke test at 504x252 (W=504, H=252) — non-standard resolution so we must pass
# interpolate_pos_encoding=True.
_dummy = torch.zeros(1, 3, 252, 504, device=device)
with torch.no_grad():
    _out = dino(pixel_values=_dummy, interpolate_pos_encoding=True)
    _cls = _out.last_hidden_state[:, 0, :]
assert _cls.shape == (1, 384), f"Expected (1, 384), got {_cls.shape}"
print(f"DINOv2 smoke test passed — CLS shape: {_cls.shape}")
del _dummy, _out, _cls

## 4. Load Training CSV

In [ ]:
df_raw = pd.read_csv(TRAIN_CSV)
df_raw["image_id"] = df_raw["sample_id"].str.split("__").str[0]

df_wide = df_raw.pivot_table(
    index=["image_id", "image_path"],
    columns="target_name",
    values="target",
).reset_index()

Y_all_real = df_wide[TARGETS].values.astype(np.float32)      # (N_real, 5)
train_image_ids_all = df_wide["image_id"].values              # (N_real,)

print(f"Training images: {len(df_wide)}")
print(f"Y_all_real shape: {Y_all_real.shape}")
print(df_wide.head(3))

## 5. Extract DINOv2 Features — Real Training Images

If `cfg.use_flip4x=True`, each image is extracted in 4 orientations (identity, hflip, vflip, 180°) to match Victor's baseline. `image_group_ids` keeps track of which rows come from the same source image so GroupKFold doesn't leak between splits.

In [ ]:
# 504x252 = (W x H); torchvision Resize takes (H, W) = (252, 504)
_dino_transform = T.Compose([
    T.Resize((252, 504)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def extract_features(image_paths, model, transform):
    """Return (N, 384) float32 array of CLS-token features."""
    feats = []
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB")
        x   = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out  = model(pixel_values=x, interpolate_pos_encoding=True)
            feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
        feats.append(feat)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


_ORIENTATIONS = [
    lambda img: img,                      # identity
    lambda img: TF.hflip(img),            # horizontal flip
    lambda img: TF.vflip(img),            # vertical flip
    lambda img: TF.hflip(TF.vflip(img)),  # 180° (hflip + vflip)
]


def extract_features_augmented(image_paths, model, transform):
    """Return (N*4, 384). Each image appears 4 times (id, hflip, vflip, 180°)."""
    feats = []
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB")
        for flip_fn in _ORIENTATIONS:
            x = transform(flip_fn(img)).unsqueeze(0).to(device)
            with torch.no_grad():
                out  = model(pixel_values=x, interpolate_pos_encoding=True)
                feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
            feats.append(feat)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


real_paths = [os.path.join(TRAIN_IMG_DIR, f"{iid}.jpg") for iid in train_image_ids_all]
N_real = len(real_paths)

if cfg.use_flip4x:
    print(f"Extracting real features with 4-orientation augmentation ({N_real} images x 4)...")
    X_real           = extract_features_augmented(real_paths, dino, _dino_transform)
    Y_real           = np.repeat(Y_all_real, 4, axis=0)
    real_image_ids   = np.repeat(train_image_ids_all, 4)
    real_group_ids   = np.repeat(np.arange(N_real), 4)
else:
    print(f"Extracting real features (no flip augmentation) for {N_real} images...")
    X_real           = extract_features(real_paths, dino, _dino_transform)
    Y_real           = Y_all_real
    real_image_ids   = train_image_ids_all
    real_group_ids   = np.arange(N_real)

print(f"X_real:         {X_real.shape}")
print(f"Y_real:         {Y_real.shape}")
print(f"Unique sources: {len(np.unique(real_group_ids))}")

## 6. Extract DINOv2 Features — Test Images

In [ ]:
df_test_raw = pd.read_csv(TEST_CSV)
df_test_raw["image_id"] = df_test_raw["sample_id"].str.split("__").str[0]
df_test_unique = df_test_raw.drop_duplicates("image_id").copy()

test_image_ids   = df_test_unique["image_id"].values
test_image_paths = [os.path.join(TEST_IMG_DIR, f"{iid}.jpg") for iid in test_image_ids]

print(f"Extracting features for {len(test_image_paths)} test images...")
X_test = extract_features(test_image_paths, dino, _dino_transform)
print(f"X_test: {X_test.shape}")

## 7. GroupKFold Train/Val Split (Real Images)

Done BEFORE synthetic extraction so that we can filter out synthetic images whose source falls into the validation fold (leakage prevention). Grouping is by source-image index, so all 4 orientations of the same image stay together.

In [ ]:
gkf = GroupKFold(n_splits=cfg.n_splits)
splits = list(gkf.split(X_real, groups=real_group_ids))
train_idx_real, val_idx_real = splits[cfg.val_fold]

train_groups = set(real_group_ids[train_idx_real].tolist())
val_groups   = set(real_group_ids[val_idx_real].tolist())
assert train_groups.isdisjoint(val_groups), "Group leakage: source images shared across folds!"

# Set of original image IDs in the training fold — used to filter synthetic sources
train_source_ids = set(real_image_ids[train_idx_real].tolist())
val_source_ids   = set(real_image_ids[val_idx_real].tolist())

print(f"Train rows (real, flipped): {len(train_idx_real)}")
print(f"Val   rows (real, flipped): {len(val_idx_real)}")
print(f"Unique sources — train: {len(train_source_ids)}  val: {len(val_source_ids)}")

## 8. DA-Fusion — Load & Filter Synthetic Images

Only synthetics whose `source_image` is in the training fold are kept. This prevents the model from training on variants of validation images.

In [ ]:
df_synth_full = pd.read_csv(cfg.synthetic_csv)
df_synth = df_synth_full[df_synth_full["is_synthetic"].astype(bool)].copy()
print(f"Synthetic rows in CSV: {len(df_synth)}")


def _source_id_from_path(p):
    """'train/ID1234.jpg' -> 'ID1234'."""
    return Path(str(p)).stem


df_synth["source_id"] = df_synth["source_image"].apply(_source_id_from_path)
df_synth = df_synth[df_synth["source_id"].isin(train_source_ids)].reset_index(drop=True)
print(f"After leakage filter (source in train fold): {len(df_synth)}")
print(df_synth[["image_path", "source_image", "source_id"]].head())

## 9. DA-Fusion — Extract DINOv2 Features on Synthetic Images

Synthetic images are extracted without the 4-orientation flip (they are already data augmentations — no need to augment augmentations).

In [ ]:
def _resolve_synth_path(image_path_field):
    """Synthetic CSV image_path may be absolute, repo-relative, or filename-only.
    Try multiple candidates rooted at cfg.synthetic_dir."""
    candidates = [
        os.path.join(cfg.synthetic_dir, image_path_field),
        os.path.join(cfg.synthetic_dir, os.path.basename(image_path_field)),
        os.path.join(cfg.synthetic_dir, Path(image_path_field).name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not locate synthetic image: {image_path_field} under {cfg.synthetic_dir}")


synth_paths = [_resolve_synth_path(p) for p in df_synth["image_path"].values]
print(f"Extracting features for {len(synth_paths)} synthetic images...")
X_synth = extract_features(synth_paths, dino, _dino_transform)
Y_synth = df_synth[TARGETS].values.astype(np.float32)

# Each synthetic inherits its source's group_id so GroupKFold-style grouping
# stays consistent (though we're past the split now, this matters if you
# re-split later, e.g. for CV).
source_to_group = {iid: int(grp)
                   for iid, grp in zip(real_image_ids, real_group_ids)}
synth_group_ids = np.array([source_to_group[sid] for sid in df_synth["source_id"].values])

print(f"X_synth: {X_synth.shape}")
print(f"Y_synth: {Y_synth.shape}")
print(f"synth groups: min={synth_group_ids.min()} max={synth_group_ids.max()} unique={len(np.unique(synth_group_ids))}")

## 10. Build Final Training Set

`X_train_daf = [real-train-flipped]  +  [synthetic filtered by train source]`

Validation is real images only (val fold), no synthetic — keeps Kaggle-leaderboard-comparable.

In [ ]:
X_train = np.vstack([X_real[train_idx_real], X_synth]).astype(np.float32)
Y_train_raw = np.vstack([Y_real[train_idx_real], Y_synth]).astype(np.float32)

X_val       = X_real[val_idx_real]
Y_val_raw   = Y_real[val_idx_real]

print(f"Final X_train: {X_train.shape}  (real-train={len(train_idx_real)} + synth={len(X_synth)})")
print(f"Final X_val:   {X_val.shape}")
print(f"Y range train: [{Y_train_raw.min():.2f}, {Y_train_raw.max():.2f}]")

## 11. MinMaxScaler on Y_train

Kept for parity with Victor's baseline even though plain ERM trains on raw targets; scaler is used only to report scaled metrics if needed later.

In [ ]:
y_scaler = MinMaxScaler()
y_scaler.fit(Y_train_raw)
print("Scaler data_min_:", y_scaler.data_min_.round(2))
print("Scaler data_max_:", y_scaler.data_max_.round(2))

## 12. Model Architecture — Encoder, Head, BiomassModel

In [ ]:
class Encoder(nn.Module):
    """384 → 128 → 32 (GELU, Dropout)."""
    def __init__(self, input_dim=384, latent_dim=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, latent_dim),
        )
    def forward(self, x):
        return self.net(x)


class Head(nn.Module):
    """32 → 32 → 5 (GELU, Dropout)."""
    def __init__(self, latent_dim=32, output_dim=5, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, output_dim),
        )
    def forward(self, z):
        return self.net(z)


class BiomassModel(nn.Module):
    def __init__(self, input_dim=384, latent_dim=32, output_dim=5, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim, dropout)
        self.head    = Head(latent_dim, output_dim, dropout)
    def encode(self, x):
        return self.encoder(x)
    def forward(self, x):
        return self.head(self.encode(x))


print("BiomassModel defined.")
_tmp = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout)
n_params = sum(p.numel() for p in _tmp.parameters())
print(f"Parameters: {n_params:,}")
del _tmp

## 13. Loss, Metrics, Dataset, Eval

In [ ]:
WEIGHT_VECTOR = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
_LOSS_WEIGHTS = torch.tensor(WEIGHT_VECTOR, dtype=torch.float32)


def _weighted_smooth_l1(pred, target, weights, beta=1.0):
    """Per-target weighted SmoothL1, averaged over the batch."""
    loss_per = nn.functional.smooth_l1_loss(pred, target, beta=beta, reduction="none")
    return (loss_per * weights.to(pred.device)).mean()


def weighted_global_r2(y_true, y_pred):
    """Kaggle competition metric: weighted global R² across all (image, target) pairs."""
    w  = WEIGHT_VECTOR
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    return {t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i, t in enumerate(TARGETS)}


class BiomassDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]


@torch.no_grad()
def eval_epoch(model, loader, eval_device):
    model.eval()
    total_loss, all_pred, all_true = 0.0, [], []
    for X, y in loader:
        X, y = X.to(eval_device), y.to(eval_device)
        pred = model(X)
        total_loss += _weighted_smooth_l1(pred, y, _LOSS_WEIGHTS).item() * len(X)
        all_pred.append(pred.cpu().numpy())
        all_true.append(y.cpu().numpy())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    r2   = weighted_global_r2(y_true, y_pred)
    rmse = rmse_per_target(y_true, y_pred)
    return total_loss / len(loader.dataset), r2, rmse, y_pred, y_true


print("Loss, metrics, dataset, eval_epoch defined.")

## 14. Training — Plain ERM

Standard AdamW + cosine-annealing LR with weighted SmoothL1 loss on raw targets. No CEMS, no synthetic-downweighting — all rows in `X_train` (real + synthetic) contribute equally.

In [ ]:
train_ds = BiomassDataset(X_train, Y_train_raw)
val_ds   = BiomassDataset(X_val,   Y_val_raw)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=64,              shuffle=False, num_workers=0)

torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
model     = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.lr / 100)

history = {k: [] for k in ["train_loss", "val_loss", "val_r2", "val_rmse"]}
best_val_r2, best_state = -float("inf"), None

print(f"Training plain ERM  —  epochs={cfg.epochs}  lr={cfg.lr}  wd={cfg.weight_decay}")
for epoch in range(1, cfg.epochs + 1):
    model.train()
    ep_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(X)
        loss = _weighted_smooth_l1(pred, y, _LOSS_WEIGHTS)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item() * len(X)
    scheduler.step()
    tr = ep_loss / len(train_loader.dataset)

    val_loss, val_r2, val_rmse, _, _ = eval_epoch(model, val_loader, device)
    history["train_loss"].append(tr)
    history["val_loss"].append(val_loss)
    history["val_r2"].append(val_r2)
    history["val_rmse"].append(val_rmse)

    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 10 == 0:
        rmse_str = "  ".join(f"{t.split('_')[1]}:{v:.2f}" for t, v in val_rmse.items())
        print(f"  ep {epoch:3d}  tr={tr:.4f}  val_loss={val_loss:.4f}  val_R²={val_r2:.4f}  [{rmse_str}]")

best_epoch = int(np.argmax(history["val_r2"])) + 1
print(f"\nBest val R² = {best_val_r2:.4f} at epoch {best_epoch}")

## 15. Final Validation Metrics

In [ ]:
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
_, val_r2_best, val_rmse_best, _, _ = eval_epoch(model, val_loader, device)

print("=" * 55)
print(f"Final val weighted R²: {val_r2_best:.4f}")
print("=" * 55)
print("RMSE per target:")
for t, v in val_rmse_best.items():
    print(f"  {t:<18}: {v:.4f}")
print("=" * 55)

## 16. Test Inference

In [ ]:
model.eval()
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
with torch.no_grad():
    test_preds = model(X_test_t).cpu().numpy()
test_preds = np.clip(test_preds, 0.0, None)   # biomass cannot be negative

print(f"test_preds shape: {test_preds.shape}")
print(f"test_preds range: [{test_preds.min():.2f}, {test_preds.max():.2f}]")
for i in range(min(3, len(test_image_ids))):
    print(f"  {test_image_ids[i]}: {dict(zip(TARGETS, test_preds[i].round(2)))}")

## 17. Format Predictions as Submission CSV

In [ ]:
def prepare_submission(test_csv_path, predictions, image_ids):
    """Returns long-format DataFrame with columns [sample_id, target]."""
    df_t = pd.read_csv(test_csv_path)
    pred_dict = {
        img_id: {col: float(val) for col, val in zip(TARGETS, pred_row)}
        for img_id, pred_row in zip(image_ids, predictions)
    }
    def _get_pred(row):
        img_id      = row["sample_id"].split("__")[0]
        target_name = row["target_name"]
        val = pred_dict.get(img_id, {}).get(target_name, 0.0)
        return max(0.0, val)
    df_t["target"] = df_t.apply(_get_pred, axis=1)
    return df_t[["sample_id", "target"]]


submission = prepare_submission(TEST_CSV, test_preds, test_image_ids)
out_path = os.path.join(cfg.output_dir, "submission.csv")
submission.to_csv(out_path, index=False)

print(f"Submission saved to: {out_path}")
print(f"Rows: {len(submission)}")
print(submission.head(10))
print("\nTarget value stats:")
print(submission["target"].describe().round(3))
print("\nWorking dir:", os.listdir(cfg.output_dir))